# Google ADK Agent — Building Agents with Google's Agent Development Kit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/83-google-adk-agent/google_adk_agent_workbook.ipynb)

Google's **Agent Development Kit (ADK)** is a high-level framework for building agents powered by Gemini models.
Where LangGraph asks you to declare a `StateGraph` with explicit nodes and edges, ADK asks you to declare an
`LlmAgent` with tools — the framework runs the reasoning loop for you.

**What you'll learn:**
- How ADK's `LlmAgent` + `Runner` replaces LangGraph's `StateGraph` loop
- How to define plain Python functions as ADK tools
- How to iterate `runner.run_async()` events to capture the final response
- How the ADK tool-dispatch cycle differs from LangGraph's node graph

## Workshop Roadmap

| # | Section | Exercise |
|---|---------|----------|
| 1 | Framework Landscape | Compare ADK vs LangGraph |
| 2 | LlmAgent and Runner | Create agent + session |
| 3 | Running the Agent | Query the agent |
| 4 | Tool Calls and the Agent Loop | Multi-turn tool dispatch |
| 5 | Contrast with LangGraph | Side-by-side comparison |

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "google-adk", "google-genai", "python-dotenv"],
        check=True,
    )
    print("Colab install complete.")
else:
    print("Local environment — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("GOOGLE_API_KEY", "")
print("API key loaded." if key else "WARNING: GOOGLE_API_KEY not set.")

## Part 1 — Framework Landscape

**LangGraph** asks you to own the agent loop:
- Define a `TypedDict` state
- Build a `StateGraph` with explicit nodes and edges
- Wire conditional edges for tool dispatch
- `compile()` and `invoke()`

**ADK** internalises the loop:
- Declare an `LlmAgent` with a model, tools, and an instruction
- Create a `Runner` and `InMemorySessionService`
- Call `runner.run_async()` — ADK handles tool dispatch, result injection, and loop termination

| Dimension | LangGraph | Google ADK |
|-----------|-----------|------------|
| Graph / loop control | Explicit StateGraph | Framework-managed loop |
| Tool definition | `@tool` decorator or `StructuredTool` | Plain Python function + docstring |
| State management | Typed state dict, reducers | Session managed by `SessionService` |
| Multi-agent support | Subgraphs, supervisor patterns | `MultiAgent` / agent teams |

> **ADK is to LangGraph as FastAPI is to raw WSGI — higher abstraction, less control, faster to prototype.**

In [ ]:
# ADK reads plain Python docstrings to build the tool schema — no decorators needed.

KNOWLEDGE_BASE: dict = {
    "python":     "Python is a high-level, dynamically typed programming language.",
    "langgraph":  "LangGraph is a library for building stateful multi-actor LLM applications.",
    "google-adk": "Google ADK simplifies building AI agents powered by Gemini models.",
    "openai":     "OpenAI provides large language model APIs including the GPT series.",
}


def lookup_topic(topic: str) -> str:
    '''Look up a topic in the knowledge base.

    Args:
        topic: The topic name to search for (e.g. 'python', 'langgraph').

    Returns:
        A description of the topic, or a not-found message.
    '''
    result = KNOWLEDGE_BASE.get(topic.lower().strip())
    return result if result else f"No entry found for '{topic}'."


def add_topic(topic: str, description: str) -> str:
    '''Add a new topic to the knowledge base.

    Args:
        topic: The topic name to store.
        description: A short description of the topic.

    Returns:
        A confirmation message.
    '''
    KNOWLEDGE_BASE[topic.lower().strip()] = description
    return f"Stored: '{topic}' -> '{description}'"


print("Tools defined. Knowledge base entries:", list(KNOWLEDGE_BASE.keys()))

## Part 2 — LlmAgent and Runner

Three ADK objects work together:

| Object | Role |
|--------|------|
| `LlmAgent` | Wraps the model + tools + system instruction |
| `Runner` | Manages the agent reasoning loop and session I/O |
| `InMemorySessionService` | Stores conversation turns in-process (no DB needed) |

The agent receives a `genai_types.Content` message and yields `Event` objects.
We check `event.is_final_response()` to know when the agent is done.

In [ ]:
import asyncio
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

agent = LlmAgent(
    model="gemini-2.0-flash",
    name="knowledge-agent",
    instruction=(
        "You are a helpful assistant with a technology knowledge base. "
        "Use lookup_topic to find information and add_topic to store new facts."
    ),
    tools=[lookup_topic, add_topic],
)

print("Agent created:", agent.name)

## Part 3 — Running the Agent

`Runner.run_async()` is an async generator that yields `Event` objects as the agent works.
Each event represents one step: a tool call, a tool result, or the final text response.

We iterate the events and call `event.is_final_response()` to detect completion.
The final response text lives at `event.content.parts[0].text`.

Because `run_async()` is a coroutine, we wrap it in `asyncio.run()` for notebook execution.

In [ ]:
APP_NAME = "workshop-agent"
USER_ID = "learner-01"
SESSION_ID = "session-01"


async def run_query(query: str) -> str:
    session_service = InMemorySessionService()
    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )

    user_message = genai_types.Content(
        role="user",
        parts=[genai_types.Part(text=query)],
    )

    final_response = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final_response = event.content.parts[0].text
    return final_response


# Test single query
answer = asyncio.run(run_query("What is LangGraph?"))
print("Q: What is LangGraph?")
print("A:", answer)

## Part 4 — Tool Calls and the Agent Loop

When the model decides it needs data, it emits a **tool-call event**.
ADK intercepts it, dispatches the Python function, and injects the result back into the model's context.
The loop continues until the model stops emitting tool calls.

```
User message
    ↓
LlmAgent (model=gemini-2.0-flash)
    ↓ tool call
ADK dispatches to lookup_topic()
    ↓ result
LlmAgent synthesizes response
    ↓
Final response → Event.is_final_response()
```

You never write this dispatch loop yourself — ADK owns it entirely.

In [ ]:
# Run multiple queries to observe tool dispatch across turns.

queries = [
    "What is the Google ADK?",
    "Add this fact: 'redis' means 'Redis is an in-memory key-value store'.",
    "Now look up redis.",
]

for query in queries:
    print(f"\nQ: {query}")
    answer = asyncio.run(run_query(query))
    print(f"A: {answer}")

## Part 5 — Contrast with LangGraph

The same "look up a topic and answer" task in both frameworks:

**LangGraph approach:**
1. Define `TypedDict` state
2. Write a node function that manually calls the tool
3. Build `StateGraph`, add nodes, add edges
4. Compile and invoke

**ADK approach:**
1. Define the tool function
2. Create `LlmAgent` with the tool
3. Call `runner.run_async()`

**Trade-offs:**

| Concern | LangGraph | ADK |
|---------|-----------|-----|
| Lines of code | ~30–50 | ~10–15 |
| Loop visibility | Full — you write it | Hidden — framework owns it |
| Debugging | Trace each node | Event stream only |
| Complex branching | First-class (conditional edges) | Limited |
| Time to prototype | Longer | Faster |

In [ ]:
# LangGraph equivalent — for comparison only, not executed.
#
# from langgraph.graph import StateGraph, END, START
# from typing import TypedDict
#
# class State(TypedDict):
#     query: str
#     answer: str
#
# def agent_node(state):
#     # manual tool dispatch, manual loop
#     ...
#
# builder = StateGraph(State)
# builder.add_node("agent", agent_node)
# builder.add_edge(START, "agent")
# builder.add_edge("agent", END)
# app = builder.compile()

# ADK equivalent (what we built above):
print("ADK agent in 5 lines:")
print('''
agent = LlmAgent(
    model="gemini-2.0-flash",
    tools=[lookup_topic, add_topic],
    instruction="..."
)
''')
print("LangGraph needs explicit StateGraph, nodes, edges, and tool dispatch logic.")
print("ADK handles all that internally — trade-off: less control, faster prototyping.")

## Exercises

**Exercise 1 — Expand the knowledge base**

Use `add_topic` (via the agent) to add 3 more technology entries of your choice.
Then call `lookup_topic` to verify each one was stored correctly.

**Exercise 2 — Change the response format**

Modify the `agent`'s `instruction` to include: *"Always respond using bullet points only."*
Re-run a query and observe how the format changes.
Think about: what does this tell you about how the instruction affects model output?

In [ ]:
# Exercise 1 — Add and retrieve new topics via the agent.

topics_to_add = [
    ("docker",     "Docker is a platform for containerizing applications."),
    ("kubernetes", "Kubernetes orchestrates containerized workloads at scale."),
    ("fastapi",    "FastAPI is a modern Python web framework for building APIs."),
]

for topic, description in topics_to_add:
    prompt = f"Add this: '{topic}' means '{description}'."
    answer = asyncio.run(run_query(prompt))
    print(f"Added {topic}: {answer[:80]}...")

# Verify retrieval
verify_query = "What is Docker?"
print(f"\nVerify: {verify_query}")
print(asyncio.run(run_query(verify_query)))

---

**Next:** [84 — Haystack Pipeline](../84-haystack-pipeline/haystack_pipeline_workbook.ipynb) — stateless DAG components with named port wiring.